# Course Outcome 1 (CO1) - Student Attendance Analysis System

## 1. Problem Statement
An educational institution needs a Python-based program to analyze student attendance. The system should accept student name, total classes conducted, and total classes attended. It needs to calculate individual attendance percentages, identify students with attendance below 75%, find the student(s) with the highest attendance, and calculate the overall class attendance metrics.

## 2. Divide into Parts/Modules
To solve this problem cleanly, the system is divided into the following functional parts:
1. **Core Operations Logic**: Functions to register and manage student records (`add_student`).
2. **Computational Logic**: Functions to compute attendance percentage, identify students below the 75% threshold, find the highest attendance records, and calculate overall class metrics.
3. **Data Store & Input Setup**: Defines the structure for storing student records (dictionaries).
4. **Driver Logic (Main)**: Interactive console menu loop to prompt user choices, validate inputs, and display formatted analysis results.

## 3. Abstraction
Abstraction helps focus only on the essential data required to solve the problem and ignores unnecessary details.

### Needed Details (Essential Information):
* **Student Name**: String key to identify each student uniquely.
* **Classes Conducted**: Total lectures conducted for the student.
* **Classes Attended**: Total lectures attended by the student.
* **Attendance Threshold (75%)**: Numeric boundary to flag low attendance.

### Useless Details (Irrelevant Information):
* **Student Background**: Grades, enrollments, age, gender, contact info, or email.
* **Course Elements**: Names of instructors, subject codes, classroom locations, or schedules.
* **Payment/Institutional Fees**: Tuition payment status, library dues, hostel details.

## 4. Algorithm
The algorithm is structured as follows:
1. Initialize an empty dictionary `student_db`.
2. Display a menu loop with 6 options (Add Student, View Report, View Low Attendance, Find Highest Attendance, View Overall Metrics, Exit).
   - Note: At the start of every loop iteration, the system displays the available branches, student count per branch, and grand total.
3. On adding a student:
   - Check name is not empty.
   - Check conducted classes > 0.
   - Check 0 <= attended <= conducted.
   - Store name as key, and conducted/attended as a nested dictionary value.
4. On displaying report/statistics:
   - For each student, compute attendance: Percentage = (Attended / Conducted) * 100.
   - Print individual results in a clean formatted grid.
5. On low attendance:
   - Print all students whose computed percentage is < 75.0%.
6. On highest attendance:
   - Locate the maximum percentage value across the dictionary.
   - Retrieve and display all students matching the maximum percentage (handling ties).
7. On overall class metrics (Option 5):
   - Prompt for stream filter. If provided, compute and print metrics (Number of Students, Class Average Percentage, and Aggregate Attendance Rate) for that specific stream.
   - If no filter is provided (user presses Enter), compute and print metrics stream-wise for every stream in the database (sorted by student count descending, then alphabetically), followed by the overall metrics for all streams combined.


## 5. Core Logic
Defines functions for inserting records, calculating attendance, and finding statistics.

In [6]:
def add_student(student_db, name, stream, conducted, attended):
    name = name.strip()
    stream = stream.strip()
    if not name:
        raise ValueError("Student name cannot be empty.")
    if not stream:
        raise ValueError("Stream cannot be empty.")
    if conducted <= 0:
        raise ValueError("Total conducted classes must be greater than zero.")
    if attended < 0:
        raise ValueError("Total attended classes cannot be negative.")
    if attended > conducted:
        raise ValueError("Attended classes cannot exceed conducted classes.")
    
    student_db[name] = {
        "stream": stream,
        "conducted": conducted,
        "attended": attended
    }

def calculate_percentage(conducted, attended):
    if conducted == 0:
        return 0.0
    return (attended / conducted) * 100.0

def get_low_attendance_students(student_db, threshold=75.0, stream=None):
    low_list = []
    for name, info in student_db.items():
        if stream and info.get("stream", "").strip().lower() != stream.strip().lower():
            continue
        pct = calculate_percentage(info["conducted"], info["attended"])
        if pct < threshold:
            low_list.append((name, info.get("stream", "N/A"), pct))
    return low_list

def get_highest_attendance_students(student_db, stream=None):
    if not student_db:
        return []
    
    filtered_db = {}
    for name, info in student_db.items():
        if stream and info.get("stream", "").strip().lower() != stream.strip().lower():
            continue
        filtered_db[name] = info
        
    if not filtered_db:
        return []
        
    max_pct = -1.0
    highest_list = []
    for name, info in filtered_db.items():
        pct = calculate_percentage(info["conducted"], info["attended"])
        if pct > max_pct:
            max_pct = pct
            highest_list = [(name, info.get("stream", "N/A"), pct)]
        elif pct == max_pct:
            highest_list.append((name, info.get("stream", "N/A"), pct))
    return highest_list

def calculate_overall_attendance(student_db, stream=None):
    filtered_db = {}
    for name, info in student_db.items():
        if stream and info.get("stream", "").strip().lower() != stream.strip().lower():
            continue
        filtered_db[name] = info
        
    if not filtered_db:
        return {
            "average_percentage": 0.0,
            "overall_rate": 0.0
        }
    
    total_pct_sum = 0.0
    total_conducted = 0
    total_attended = 0
    
    for name, info in filtered_db.items():
        pct = calculate_percentage(info["conducted"], info["attended"])
        total_pct_sum += pct
        total_conducted += info["conducted"]
        total_attended += info["attended"]
        
    return {
        "average_percentage": total_pct_sum / len(filtered_db),
        "overall_rate": (total_attended / total_conducted) * 100.0 if total_conducted > 0 else 0.0
    }


## 6. Data Store & Input Setup
Defines standard initial test data records for analysis and demonstration.

In [7]:
import csv
import os

DEFAULT_STUDENT_DB = {
    "Aarav": {"stream": "CSE", "conducted": 40, "attended": 35},
    "Ananya": {"stream": "CSE", "conducted": 40, "attended": 38},
    "Manoj": {"stream": "ECE", "conducted": 40, "attended": 34},
    "Meghana": {"stream": "ECE", "conducted": 45, "attended": 39},
    "Aditya": {"stream": "AIML", "conducted": 50, "attended": 47},
    "Aishwarya": {"stream": "AIML", "conducted": 45, "attended": 33},
    "Abhinav": {"stream": "DS", "conducted": 40, "attended": 36},
    "Bhargavi": {"stream": "DS", "conducted": 45, "attended": 34},
    "Chaitanya": {"stream": "DS", "conducted": 50, "attended": 43},
    "Dhanush": {"stream": "CSE", "conducted": 40, "attended": 29},
    "Keerthi": {"stream": "MBA", "conducted": 45, "attended": 41},
    "Lokesh": {"stream": "CSIT", "conducted": 50, "attended": 37},
    "Nandini": {"stream": "AIDS", "conducted": 40, "attended": 31},
    "Rohit": {"stream": "BBA", "conducted": 45, "attended": 44},
    "Sravani": {"stream": "MECH", "conducted": 50, "attended": 35},
    "Vamsi": {"stream": "BBA", "conducted": 40, "attended": 38}
}

def load_student_db(csv_path=None):
    """
    Initializes the student database.
    If csv_path is provided and the file exists, it loads the data from the CSV file.
    Otherwise, it falls back to the default student database.
    """
    db = {}
    if csv_path and os.path.exists(csv_path):
        print(f"Loading student records from CSV: {csv_path}")
        try:
            with open(csv_path, mode='r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                if not reader.fieldnames:
                    print("Error: Empty CSV or invalid headers. Falling back to default.")
                    return DEFAULT_STUDENT_DB.copy()
                headers = {h.strip().lower(): h for h in reader.fieldnames}
                name_col = next((headers[h] for h in ['student name', 'student_name', 'name'] if h in headers), None)
                stream_col = next((headers[h] for h in ['stream', 'class', 'branch', 'department'] if h in headers), None)
                conducted_col = next((headers[h] for h in ['conducted classes', 'conducted_classes', 'conducted'] if h in headers), None)
                attended_col = next((headers[h] for h in ['attended classes', 'attended_classes', 'attended'] if h in headers), None)
                if not (name_col and stream_col and conducted_col and attended_col) and len(reader.fieldnames) >= 4:
                    name_col = reader.fieldnames[0]
                    stream_col = reader.fieldnames[1]
                    conducted_col = reader.fieldnames[2]
                    attended_col = reader.fieldnames[3]
                
                if name_col and stream_col and conducted_col and attended_col:
                    for row in reader:
                        name = row[name_col].strip()
                        stream = row[stream_col].strip()
                        if not name or not stream:
                            continue
                        try:
                            conducted = int(row[conducted_col])
                            attended = int(row[attended_col])
                            if conducted <= 0 or attended < 0 or attended > conducted:
                                print(f"Warning: Skipping invalid record for '{name}' (conducted: {conducted}, attended: {attended}).")
                                continue
                            db[name] = {"stream": stream, "conducted": conducted, "attended": attended}
                        except (ValueError, TypeError):
                            print(f"Warning: Skipping row with invalid numeric data for '{name}'.")
                            continue
                else:
                    print("Error: CSV must contain 'name', 'stream', 'conducted', and 'attended' columns.")
                    print("Falling back to default database.")
                    return DEFAULT_STUDENT_DB.copy()
            print(f"Successfully loaded {len(db)} records.")
            return db
        except Exception as e:
            print(f"Error reading CSV file: {e}")
            print("Falling back to default database.")
            return DEFAULT_STUDENT_DB.copy()
    else:
        if csv_path:
            print(f"CSV file '{csv_path}' not found. Loading default database.")
        else:
            print("No CSV file provided. Loading default database.")
        return DEFAULT_STUDENT_DB.copy()
        
student_db = load_student_db()


No CSV file provided. Loading default database.


## 7. Driver Logic (Main)
Prompts the user to select either the Automated Demo or the Interactive Play mode.

In [8]:
operation_count = 0
max_operations = 30

try:
    while True:
        operation_count += 1
        if operation_count > max_operations:
            print("Reached maximum operation limit (30) to prevent kernel hang. Exiting.")
            break
            
        # Show available branches and total students in each branch, and grand total
        if student_db:
            branch_counts = {}
            for name, info in student_db.items():
                s = info.get("stream", "N/A").strip()
                branch_counts[s] = branch_counts.get(s, 0) + 1
            
            sorted_branches = sorted(branch_counts.keys(), key=lambda x: x.upper())
            print("-" * 45)
            print("AVAILABLE BRANCHES & STUDENT COUNTS")
            print("-" * 45)
            for br in sorted_branches:
                print(f"{br:<15}: {branch_counts[br]} student(s)")
            print("-" * 45)
            print(f"Grand Total    : {len(student_db)} student(s)")
            print("-" * 45)
        else:
            print("-" * 45)
            print("No student records available.")
            print("-" * 45)
            
        print("\n--- STUDENT ATTENDANCE MENU ---")
        print("1. Add Student Record")
        print("2. View Attendance Report")
        print("3. Find Students Below 75% (Low Attendance)")
        print("4. Find Student(s) with Highest Attendance")
        print("5. Calculate Overall Class Attendance")
        print("6. Load Student Records from CSV")
        print("7. Exit")
        
        choice = input("Enter choice (1-7): ").strip()
        
        if choice == "1":
            name = input("Enter Student Name: ").strip()
            stream = input("Enter Stream (e.g., CSE, ECE, AIML, DS): ").strip()
            try:
                conducted = int(input("Enter Total Classes Conducted: "))
                attended = int(input("Enter Total Classes Attended: "))
                add_student(student_db, name, stream, conducted, attended)
                print(f"Record added successfully for {name} ({stream}).")
            except ValueError as e:
                print(f"Error: {e}")
                
        elif choice == "2":
            if not student_db:
                print("No records available.")
            else:
                stream_filter = input("Enter stream to filter by (or press Enter for all): ").strip()
                stream_filter = stream_filter if stream_filter else None
                
                print("-" * 75)
                print(f"{'Student Name':<20} {'Stream':<10} {'Conducted':<10} {'Attended':<10} {'Percentage':<12} {'Status':<10}")
                print("-" * 75)
                count = 0
                for name, info in student_db.items():
                    if stream_filter and info.get("stream", "").strip().lower() != stream_filter.lower():
                        continue
                    pct = calculate_percentage(info["conducted"], info["attended"])
                    status = "Good" if pct >= 75.0 else "Low (<75%)"
                    pct_str = f"{pct:.2f}%"
                    print(f"{name:<20} {info.get('stream', 'N/A'):<10} {info['conducted']:<10} {info['attended']:<10} {pct_str:<12} {status:<10}")
                    count += 1
                if count == 0:
                    print("No records matched the filter.")
                print("-" * 75)
                
        elif choice == "3":
            stream_filter = input("Enter stream to filter by (or press Enter for all): ").strip()
            stream_filter = stream_filter if stream_filter else None
            
            low_list = get_low_attendance_students(student_db, stream=stream_filter)
            if not low_list:
                print("No students have attendance below 75%.")
            else:
                print("-" * 55)
                print(f"{'Student Name':<20} {'Stream':<10} {'Attendance Percentage':<25}")
                print("-" * 55)
                for name, stream, pct in low_list:
                    pct_str = f"{pct:.2f}%"
                    print(f"{name:<20} {stream:<10} {pct_str:<25}")
                print("-" * 55)
                
        elif choice == "4":
            stream_filter = input("Enter stream to filter by (or press Enter for all): ").strip()
            stream_filter = stream_filter if stream_filter else None
            
            highest_list = get_highest_attendance_students(student_db, stream=stream_filter)
            if not highest_list:
                print("No records available.")
            else:
                print("-" * 55)
                print(f"{'Student Name':<20} {'Stream':<10} {'Highest Percentage':<25}")
                print("-" * 55)
                for name, stream, pct in highest_list:
                    pct_str = f"{pct:.2f}%"
                    print(f"{name:<20} {stream:<10} {pct_str:<25}")
                print("-" * 55)
                
        elif choice == "5":
            stream_filter = input("Enter stream to filter by (or press Enter for all): ").strip()
            
            if not stream_filter:
                print("-" * 45)
                print("OVERALL CLASS ATTENDANCE METRICS")
                print("-" * 45)
                
                stream_counts = {}
                for name, info in student_db.items():
                    s = info.get("stream", "N/A").strip()
                    stream_counts[s] = stream_counts.get(s, 0) + 1
                
                sorted_streams = sorted(stream_counts.keys(), key=lambda s: (-stream_counts[s], s.upper()))
                
                for s in sorted_streams:
                    stats = calculate_overall_attendance(student_db, stream=s)
                    count = stream_counts[s]
                    print(f"\n{s}")
                    print("-" * 45)
                    print(f"Number of Students:{count:>14}")
                    print(f"Class Average Percentage:      {stats['average_percentage']:.2f}%")
                    print(f"Aggregate Attendance Rate:     {stats['overall_rate']:.2f}%")
                
                total_stats = calculate_overall_attendance(student_db)
                print("\n" + "-" * 45)
                print("ALL STREAMS")
                print("-" * 45)
                print(f"Number of Students:{len(student_db):>14}")
                print(f"Class Average Percentage:      {total_stats['average_percentage']:.2f}%")
                print(f"Aggregate Attendance Rate:     {total_stats['overall_rate']:.2f}%")
                print("-" * 45)
            else:
                matched_stream = None
                for info in student_db.values():
                    s = info.get("stream", "").strip()
                    if s.lower() == stream_filter.lower():
                        matched_stream = s
                        break
                
                if not matched_stream:
                    matched_stream = stream_filter.upper()
                
                stats = calculate_overall_attendance(student_db, stream=matched_stream)
                count = sum(1 for info in student_db.values() if info.get("stream", "").strip().lower() == matched_stream.lower())
                
                print("-" * 45)
                print("OVERALL CLASS ATTENDANCE METRICS")
                print("-" * 45)
                print(f"Stream: {matched_stream}")
                print("-" * 45)
                print(f"Number of Students:{count:>14}")
                print(f"Class Average Percentage:      {stats['average_percentage']:.2f}%")
                print(f"Aggregate Attendance Rate:     {stats['overall_rate']:.2f}%")
                print("-" * 45)
                
        elif choice == "6":
            csv_path = input("Enter CSV file path: ").strip()
            new_db = load_student_db(csv_path)
            if new_db:
                student_db.clear()
                student_db.update(new_db)
                print("Database re-initialized from CSV.")
            else:
                print("Failed to load records from CSV. Database remains unchanged.")
                
        elif choice == "7":
            print("Exiting Student Attendance Analysis System. Goodbye!")
            break
        else:
            print("Invalid choice. Please select options 1-7.")
except (EOFError, Exception) as e:
    if "StdinNotImplementedError" in type(e).__name__ or isinstance(e, EOFError):
        print("Interactive session skipped in non-interactive environment.")
    else:
        raise e


---------------------------------------------
AVAILABLE BRANCHES & STUDENT COUNTS
---------------------------------------------
AIDS           : 1 student(s)
AIML           : 2 student(s)
BBA            : 2 student(s)
CSE            : 3 student(s)
CSIT           : 1 student(s)
DS             : 3 student(s)
ECE            : 2 student(s)
MBA            : 1 student(s)
MECH           : 1 student(s)
---------------------------------------------
Grand Total    : 16 student(s)
---------------------------------------------

--- STUDENT ATTENDANCE MENU ---
1. Add Student Record
2. View Attendance Report
3. Find Students Below 75% (Low Attendance)
4. Find Student(s) with Highest Attendance
5. Calculate Overall Class Attendance
6. Load Student Records from CSV
7. Exit


Enter choice (1-7):  5
Enter stream to filter by (or press Enter for all):  


---------------------------------------------
OVERALL CLASS ATTENDANCE METRICS
---------------------------------------------

CSE
---------------------------------------------
Number of Students:             3
Class Average Percentage:      85.00%
Aggregate Attendance Rate:     85.00%

DS
---------------------------------------------
Number of Students:             3
Class Average Percentage:      83.85%
Aggregate Attendance Rate:     83.70%

AIML
---------------------------------------------
Number of Students:             2
Class Average Percentage:      83.67%
Aggregate Attendance Rate:     84.21%

BBA
---------------------------------------------
Number of Students:             2
Class Average Percentage:      96.39%
Aggregate Attendance Rate:     96.47%

ECE
---------------------------------------------
Number of Students:             2
Class Average Percentage:      85.83%
Aggregate Attendance Rate:     85.88%

AIDS
---------------------------------------------
Number of Students

Enter choice (1-7):  1
Enter Student Name:  aravind
Enter Stream (e.g., CSE, ECE, AIML, DS):  DS
Enter Total Classes Conducted:  55
Enter Total Classes Attended:  50


Record added successfully for aravind (DS).
---------------------------------------------
AVAILABLE BRANCHES & STUDENT COUNTS
---------------------------------------------
AIDS           : 1 student(s)
AIML           : 2 student(s)
BBA            : 2 student(s)
CSE            : 3 student(s)
CSIT           : 1 student(s)
DS             : 4 student(s)
ECE            : 2 student(s)
MBA            : 1 student(s)
MECH           : 1 student(s)
---------------------------------------------
Grand Total    : 17 student(s)
---------------------------------------------

--- STUDENT ATTENDANCE MENU ---
1. Add Student Record
2. View Attendance Report
3. Find Students Below 75% (Low Attendance)
4. Find Student(s) with Highest Attendance
5. Calculate Overall Class Attendance
6. Load Student Records from CSV
7. Exit


Enter choice (1-7):  7


Exiting Student Attendance Analysis System. Goodbye!
